# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

In [3]:
# ==========================================
# 1. Environment & Workspace Prerequisites
# ==========================================
# ==========================================
# 1. Environment & Workspace Prerequisites
# ==========================================
import importlib.util
import sys
import re
import os
import json
import chromadb
from pathlib import Path
from typing import List, Dict, Any
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from tavily import TavilyClient

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

# Import workspace custom framework modules
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

# 1. Locate the exact absolute path of the .env file next to this notebook
notebook_dir = Path(os.getcwd())
env_file_path = notebook_dir / ".env"

print(f"Checking for .env file at: {env_file_path.resolve()}")

# 2. Force load via absolute path
if env_file_path.exists():
    load_dotenv(dotenv_path=env_file_path, override=True)
    print("✅ Success: .env file found and parsed.")
else:
    # Fallback to checking the parent directory just in case
    parent_env = notebook_dir.parent / ".env"
    if parent_env.exists():
        load_dotenv(dotenv_path=parent_env, override=True)
        print("✅ Success: .env file found in parent directory.")
    else:
        print("⚠️ Warning: .env file could not be detected at expected path locations.")

# 3. Guardrail validation check
openai_key = os.getenv("OPENAI_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")

if not openai_key or "your_" in openai_key or openai_key.strip() == "":
    raise ValueError(
        "❌ CRITICAL ERROR: 'OPENAI_API_KEY' is missing or unreadable inside your .env file! "
        "Please open your .env file in the sidebar and ensure it contains a valid token string."
    )

if not tavily_key or "your_" in tavily_key or tavily_key.strip() == "":
    raise ValueError(
        "❌ CRITICAL ERROR: 'TAVILY_API_KEY' is missing or unreadable inside your .env file!"
    )

print("🚀 Keys validated. Initializing local persistent database client connection...")

# Set up local persistent database client connection
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")


# ==========================================
# 2. Pydantic Response & Validation Schemas
# ==========================================
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to completely answer the user question.")
    description: str = Field(description="Detailed explanation justifying the evaluation decision.")

class FinalAgentResponse(BaseModel):
    natural_language_summary: str = Field(description="The conversational, user-friendly response to the query.")
    structured_data: dict = Field(description="A dictionary containing supporting raw metrics, links, or facts.")


# ==========================================
# 3. Agent Tool Definitions
# ==========================================

# FIX 2 & 3: REWRITTEN RETRIEVAL TOOL (Corrected Docstring & Inner-list Indexing Bug)
@tool
def query_vector_db(query_text: str, n_results: int = 3) -> str:
    """Queries the local ChromaDB vector store containing the video game dataset 
    for relevant information regarding game release dates, platforms, or publishers."""
    try:
        results = collection.query(
            query_texts=[query_text],
            n_results=n_results
        )
        
        # ChromaDB returns a nested structure: {"documents": [["doc1", "doc2"]]}
        documents_outer = results.get("documents", [[]])
        
        # Unpack the outer list to iterate over individual string documents safely
        if not documents_outer or len(documents_outer[0]) == 0:
            return "No matching video game documents found in local database."
            
        documents = documents_outer[0]
        return "\n\n".join([f"[Doc {i+1}]: {doc}" for i, doc in enumerate(documents)])
    except Exception as e:
        return f"Error querying vector database: {str(e)}"

# FIX 4: NEW EVALUATION TOOL IMPLEMENTATION
@tool
def evaluate_retrieved_results(question: str, retrieved_context: str) -> str:
    """Evaluates whether the retrieved internal video game documents contain enough factual
    information to answer the user's question completely, outputting a structured EvaluationReport schema."""
    try:
        # CRITERION 2 FIX: Change 'model_name' keyword argument to 'model'
        eval_llm = LLM(model="gpt-4o-mini")
        
        system_instructions = (
            "You are a strict data verification assistant. Analyze the retrieved database context "
            "and determine if it provides enough information to answer the user's question completely.\n"
            "You MUST return a JSON object adhering exactly to the EvaluationReport schema:\n"
            "{\n"
            '  "useful": true/false,\n'
            '  "description": "Your detailed reasoning here..."\n'
            "}"
        )
        
        prompt = f"User Question: {question}\n\nRetrieved Database Context:\n{retrieved_context}"
        messages = [SystemMessage(content=system_instructions), UserMessage(content=prompt)]
        raw_eval = eval_llm.invoke(messages)
        
        content_str = getattr(raw_eval, 'content', str(raw_eval)).strip()
        if content_str.startswith("```json"):
            content_str = content_str.split("```json")[1].split("```")[0].strip()
        elif content_str.startswith("```"):
            content_str = content_str.split("```")[1].split("```")[0].strip()
            
        return content_str
    except Exception as e:
        # CRITERION 2 FIX: Re-raise the tool exception instead of hiding it behind a fake fallback verdict
        raise e


@tool
def web_search(query: str) -> str:
    """Performs a live web search using Tavily to find up-to-date external information."""
    try:
        tavily = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY", ""))
        response = tavily.search(query=query, max_results=3)
        results = response.get("results", [])
        if not results:
            return "No relevant web search results found."
        return "\n\n".join([f"[Web Match]: {r['title']}\nURL: {r['url']}\nContent: {r['content']}" for r in results])
    except Exception as e:
        return f"Error executing web search: {str(e)}"


# ==========================================
# 4. Agent Class Implementation
# ==========================================
class UdaplayAgent(Agent):
    def __init__(self, model_name: str = "gpt-4o-mini"):
        """Initializes the agent with strict structural retrieval guidelines."""
        
        # FIX 5: EXPLICITLY DEFINE THE TWO-TIER RETRIEVAL DESIGN
        instructions = (
            "You are the Udaplay Project Assistant.\n\n"
            "CRITICAL PROTOCOL FLOW:\n"
            "Step 1: Always execute 'query_vector_db' first to check internal game data.\n"
            "Step 2: You MUST immediately parse those database results into the 'evaluate_retrieved_results' tool.\n"
            "Step 3: If and only if the evaluation verdict states 'useful': false, you are permitted to "
            "call 'web_search' as an external fallback. Do not skip evaluation.\n"
            "Step 4: Output your final answer as a raw JSON object strictly conforming to FinalAgentResponse:\n"
            "{\n"
            '  "natural_language_summary": "Your fluid, conversational response here...",\n'
            '  "structured_data": {"sources_used": ..., "game_facts": ...}\n'
            "}"
        )
        
        super().__init__(model_name, instructions)
        
        # FIX 6: REGISTER THE THREE MANDATORY TOOLS
        self.tools = [query_vector_db, evaluate_retrieved_results, web_search]


# ==========================================
# 5. Agent Execution Loop & Required Queries
# ==========================================
# ==========================================
# 5. Agent Execution Loop & Required Queries
# ==========================================
if __name__ == "__main__":
    print("🤖 Initializing UdaplayAgent...")
    udaplay_agent = UdaplayAgent(model_name="gpt-4o-mini")
    
    # CRITERION 4 FIX: Swap out the first test question
    required_queries = [
        "Was Mortal Kombat X released for Playstation 5?",
        "When was Pokemon Gold and Silver released?",
        "Which one was the first 3d platformer Mario game?"
    ]
    
    for user_q in required_queries:
        print(f"\n🚀 {'='*20} Invoking Agent with Query: '{user_q}' {'='*20}")
        run_object = udaplay_agent.invoke(user_q)
        
        print("\n=======================================================")
        print(f"🎉 Execution Finished for: {user_q}")
        print("=======================================================")
        
        final_state = run_object.get_final_state()
        final_messages = []
        if isinstance(final_state, dict):
            final_messages = final_state.get("messages", [])
        elif hasattr(final_state, "messages"):
            final_messages = final_state.messages

        # CRITERION 4 FIX: Explicitly iterate over messages to print tool names and outputs for tracking
        print("\n🛠️ [FULL TOOL TRACE LOG]:")
        for msg in final_messages:
            if hasattr(msg, 'type') and msg.type == 'tool' or isinstance(msg, ToolMessage):
                print(f" -> Tool Used: {getattr(msg, 'name', 'Unknown Tool')}")
                print(f" -> Tool Output Snippet: {str(msg.content)[:150]}...\n")

        if final_messages:
            raw_content = getattr(final_messages[-1], 'content', str(final_messages[-1])).strip()
            
            if raw_content.startswith("```json"):
                raw_content = raw_content.split("```json")[1].split("```")[0].strip()
            elif raw_content.startswith("```"):
                raw_content = raw_content.split("```")[1].split("```")[0].strip()
                
            print("\n🔍 Validating Final Schema Alignment...")
            try:
                parsed_json = json.loads(raw_content)
                validated_response = FinalAgentResponse(**parsed_json)
                print("🌟 SUCCESS: Object conforms to FinalAgentResponse metrics!\n")
                print(f"[Natural Language Summary]:\n{validated_response.natural_language_summary}\n")
                
                # CRITERION 4 FIX: Print the entire structured data block itself so sources ship clearly
                print("[Structured Data & Sources]:")
                print(json.dumps(validated_response.structured_data, indent=2))
            except Exception as json_err:
                print(f"⚠️ Structural Verification Failed: {str(json_err)}")
                print(f"[Raw Agent Output]:\n{raw_content}")



Checking for .env file at: /workspace/Code/project/starter/.env
✅ Success: .env file found and parsed.
🚀 Keys validated. Initializing local persistent database client connection...
🤖 Initializing UdaplayAgent...

🚀 ==================== Invoking Agent with Query: 'Was Mortal Kombat X released for Playstation 5?' ====================
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor


/home/student/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 79.4MiB/s]


[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

🎉 Execution Finished for: Was Mortal Kombat X released for Playstation 5?

🛠️ [FULL TOOL TRACE LOG]:
 -> Tool Used: query_vector_db
 -> Tool Output Snippet: "[Doc 1]: [Xbox Series X|S] Halo Infinite (2021) - The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world s...

 -> Tool Used: evaluate_retrieved_results
 -> Tool Output Snippet: "{\n  \"useful\": false,\n  \"description\": \"The retrieved database context does not provide any information regarding the release of Mortal Kombat ...

 -> Tool Used: web_search
 -> Tool Output Snippet: "[Web Match]: Mortal Kombat X - IGN\nURL: https://www.ign.com/games/mortal-kombat-x\nContent: Sep 29

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

#### Evaluate Retrieval Tool

In [ ]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

#### Game Web Search Tool

In [ ]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

### Agent

In [ ]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

In [ ]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes